In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from pydrake.all import (
    Box,
    ConstantVectorSource,
    DiagramBuilder,
    QuasidynamicMultibodyPlant,
    RigidTransform,
    Simulator,
    SpatialInertia,
    Rgba,
    Meshcat,
    Cylinder,
    RollPitchYaw,
    DiscreteTimeTrajectory,
    TrajectorySource,
    VectorLogSink,
)
from algorithms import (
    IterativeTO,
    IterativeSLSController,
    TrajOptProblem,
)
from models.allegro_hand import AddAllegroHand
IS_TEST = "BAZEL_TEST" in os.environ

In [ ]:
meshcat = Meshcat()

In [ ]:
dt = 0.02
plant = QuasidynamicMultibodyPlant(dt=dt)

joint_lb, joint_ub = AddAllegroHand(
    plant,
    friction_coefficient=0.2,
    fingertip_friction_coefficient=0.2,
    actuation_stiffness=1,
)
joint_ub[13] = 1.744

w = 0.06
rho = 10
cube = plant.AddRigidBody(SpatialInertia.SolidBoxWithDensity(rho, w, w, w))
box = Box(w, w, w)
plant.RegisterCollisionGeometry(cube, RigidTransform(), box, 0.2)
plant.RegisterVisualGeometry(cube, RigidTransform(), box, Rgba(0,1,1,1), "cube/cube")
axis = Cylinder(0.001, w * 0.8)
plant.RegisterVisualGeometry(cube, RigidTransform(RollPitchYaw([0, np.pi/2, 0]), [w/2, 0, 0]), axis, Rgba(1,0,0,1), "cube/x")
plant.RegisterVisualGeometry(cube, RigidTransform(RollPitchYaw([np.pi/2, 0, 0]), [0, w/2, 0]), axis, Rgba(0,1,0,1), "cube/y")
plant.RegisterVisualGeometry(cube, RigidTransform(RollPitchYaw([0, 0, np.pi/2]), [0, 0, w/2]), axis, Rgba(0,0,1,1), "cube/z")

plant.SetGravityVector([-50, 0, 0])
plant.Finalize()
plant.SetMeshcat(meshcat)

In [ ]:
u0 = np.deg2rad([
    0, 60, 90, 40,
    0, 10, 60, 50,
    10, 65, 50, 65,
    21, 90, 90, 70,
])
x0 = np.concatenate((u0, [0, 0, 0, 0.041, 0, 0.06]))
x_target = np.concatenate((np.zeros(16), [np.pi/2, 0, 0, 0.041, 0, 0.06]))

builder = DiagramBuilder()
plant = plant.Clone()
builder.AddSystem(plant)
source = ConstantVectorSource(u0)
builder.AddSystem(source)
builder.Connect(source.get_output_port(), plant.get_actuation_input_port())

diagram = builder.Build()
simulator = Simulator(diagram)

plant_context = diagram.GetMutableSubsystemContext(
    plant, simulator.get_context()
)
plant.SetState(plant_context, x0)
plant.set_dynamics_smoothing(plant_context, 1e-4)

meshcat.StartRecording()
simulator.AdvanceTo(0)
meshcat.StopRecording()
meshcat.PublishRecording()

In [ ]:
plant = plant.Clone()
context = plant.CreateDefaultContext()
plant.set_dynamics_smoothing(context, 1e-4)

num_steps = 50 if not IS_TEST else 10
prob = TrajOptProblem(system=plant, context=context, num_steps=num_steps)
prob.SetInitialState(x0)

x = prob.state()
u = prob.input()
Q = np.diag(np.concatenate((np.zeros(16), [10, 10, 10, 1, 1, 1])))
R = np.identity(16) * 0.1
prob.AddStageCost((x - x_target).dot(Q @ (x - x_target)) + u.dot(R @ u))
prob.AddTerminalCost(1e3 * (x - x_target).dot(Q @ (x - x_target)))

for k in range(num_steps - 1):
    diff = prob.input(k) - prob.input(k + 1)
    prob.AddCost(100 * diff.dot(diff))

for k in range(num_steps):
    if k == 0:
        prob.AddConstraint(prob.input(0), lb=u0, ub=u0)
    else:
        prob.AddConstraint(prob.input(k), lb=joint_lb, ub=joint_ub)

prob.SetLinearCollisionConstraint(enable=True, distance_threshold=0.01)

In [ ]:
us_init = np.repeat(u0[np.newaxis, :], repeats=num_steps, axis=0)
traj = IterativeTO(
    prob,
    us_init=us_init,
    trust_region_covar=np.identity(38) * 0.05**-2,
    use_contact_trust_region=False,
    use_true_dynamics_for_rollout=True,
    max_iters=50,
)

us = traj.us
u_traj = DiscreteTimeTrajectory(np.arange(len(us)) * dt, us[:, :, np.newaxis])
u_traj_source = TrajectorySource(u_traj)

builder = DiagramBuilder()
plant = plant.Clone()
builder.AddSystem(plant)
builder.AddSystem(u_traj_source)
logger = VectorLogSink(22)
builder.AddSystem(logger)
builder.Connect(u_traj_source.get_output_port(), plant.get_actuation_input_port())
builder.Connect(plant.get_state_output_port(), logger.get_input_port())

diagram = builder.Build()
simulator = Simulator(diagram)

plant_context = diagram.GetMutableSubsystemContext(
    plant, simulator.get_context()
)
plant.SetState(plant_context, x0)
plant.set_dynamics_smoothing(plant_context, 0.0)

meshcat.StartRecording()
simulator.AdvanceTo(num_steps * dt)
meshcat.StopRecording()
meshcat.PublishRecording()

plt.figure(figsize=(15, 7), constrained_layout=True)
plt.suptitle("IterativeTO")
for i in range(6):
    plt.subplot(2, 3, i + 1)
    dof = i + 16

    ts = np.arange(num_steps + 1) * dt
    plt.plot(ts, traj.xs[:, dof], label="Nominal trajectory")

    log = logger.FindLog(simulator.get_context())
    plt.plot(log.sample_times(), log.data()[dof], label="True dynamics rollout")

    plt.axhline(y=x_target[dof], color="k", linestyle="--", linewidth=1.0)

    plt.xlabel("$t$")
    plt.ylabel("${}_o$".format(["roll", "pitch", "yaw", "x", "y", "z"][i]))
    plt.legend()
    plt.xlim([0, num_steps * dt])
plt.show()

In [ ]:
prob.SetLinearCollisionConstraint(enable=False)

Q_bar = np.identity(22) * 1e1
R_bar = np.identity(16) * 1e1
Qf_bar = np.identity(22) * 1e1
prob.SetUncertaintyTubeCost(Q_bar=Q_bar, R_bar=R_bar, Qf_bar=Qf_bar)

controller = IterativeSLSController(
    prob,
    us_init=traj.us,
    delta_cap=1e-8,
    max_outer_iters=1,
    # slack_penalty=1,
)

builder = DiagramBuilder()
plant = builder.AddSystem(plant.Clone())
controller = builder.AddSystem(controller)
logger = builder.AddSystem(VectorLogSink(22))
builder.Connect(plant.get_state_output_port(), controller.get_input_port())
builder.Connect(controller.get_output_port(), plant.get_actuation_input_port())
builder.Connect(plant.get_state_output_port(), logger.get_input_port())

diagram = builder.Build()
simulator = Simulator(diagram)

plant_context = diagram.GetMutableSubsystemContext(
    plant, simulator.get_context()
)
plant.SetState(plant_context, x0)
plant.set_dynamics_smoothing(plant_context, 0.0)
plant.set_geometry_smoothing(plant_context, 0.0)

meshcat.StartRecording()
simulator.AdvanceTo(num_steps * dt)
meshcat.StopRecording()
meshcat.PublishRecording()

In [ ]:
plt.figure(figsize=(15, 7), constrained_layout=True)
plt.suptitle("IterativeSLS")
for i in range(6):
    plt.subplot(2, 3, i + 1)
    dof = i + 16

    ts = np.arange(num_steps + 1) * dt
    xs = controller.xs
    xs_lb = controller.xs_lb
    xs_ub = controller.xs_ub
    # plt.plot(ts, xs[:, dof], label="Nominal trajectory")
    # plt.plot(ts, xs_lb[:, dof])
    # plt.plot(ts, xs_ub[:, dof])
    plt.fill_between(ts, xs_lb[:, dof], xs_ub[:, dof], alpha=0.3)

    log = logger.FindLog(simulator.get_context())
    # plt.plot(log.sample_times(), log.data()[dof], label="True dynamics rollout")

    plt.axhline(y=x_target[dof], color="k", linestyle="--", linewidth=1.0)

    plt.xlabel("$t$")
    plt.ylabel("${}_o$".format(["roll", "pitch", "yaw", "x", "y", "z"][i]))
    # plt.legend()
    plt.xlim([0, num_steps * dt])
plt.show()